# Assignment 2

## Part 1: Interacting with APIs

`https://en.wikipedia.org/w/api.php` is the endpoint for the English Wikipedia API. It can be used to access a lot of different information from and about English Wikipedia (or English *language* Wikipedia, as some prefer to say).

Complete all the exercises using API calls to this endpoint using the `requests` library.

- **Hint 1**: You may want to work on your browser first to test and prototype, then come back to Python for implementation.

- **Hint 2**: When sending requests from Python, you should use the `&format=json` argument to make sure your result is easily convertible to a Python object.

In [2]:
# let's import requests right here for your convenience
# run me!

import requests

### Question 1 (2.5 points)


In this exercise, you will find the the total number of page views (i.e. visits) to the Wikipedia page for the city of Boston in the last 30 days.

Do this in two steps:

1. Write a function called `get_view_totals` that can get the number of total page views in the last 30 days for any given page title.

2. Use the function on the title `Boston` and store the result into a variable called `total_views_boston`


Some tips:
- You can go to the endpoint provided in the beginning, follow the links to `query` then `pageviews` to see example API calls.
- You will see that the API, by default, gives the last 60 days. But, it also explains how you can specify a certain parameter to limit the results to 30 days.




In [3]:
def get_view_totals(page_title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "pageviews",
        "titles": page_title
    }
    response = requests.get(url, params=params)
    data = response.json()
    pageviews = next(iter(data['query']['pages'].values()))['pageviews']
    total_views = sum(view for view in pageviews.values() if view)
    return total_views

total_views_boston = get_view_totals("Boston")
print(total_views_boston)

262256


### Question 2 (2.5 points)

Many Wikipedia pages have versions in many different languages. By visiting [the page for Boston](https://en.wikipedia.org/wiki/Boston), one can see (on the upper right corner) that it has pages in 172 different Wikipedia languages other than English (at the time this writing).

Use the API to get that number for Boston. In the endpoint, you can follow links to `query` and then `langlinkscount` to see an example.

Do this in two steps:

1. Write a function called `get_langlinkscount` that returns the number of other languages for any given page title.

2. Use the function on the title `Boston` to get the result. Store this in a variable called `langlinkscount_boston`



In [4]:
def get_langlinkscount(page_title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "langlinks",
        "titles": page_title,
        "lllimit": "max"
    }
    response = requests.get(url, params=params)
    data = response.json()
    langlinks = next(iter(data['query']['pages'].values())).get("langlinks", [])
    return len(langlinks)

langlinkscount_boston = get_langlinkscount("Boston")
print(langlinkscount_boston)

172


## Part 2: Scraping Web Pages

In this part, you will scrape the following page: `https://en.wikipedia.org/w/index.php?title=Massachusetts&oldid=1263937561`

The page is the Wikipedia page for Massachusetts, but fixed at a specific version (or a specific "revision", as Wikipedia calls it) so that everyone gets the same results no matter what time the scraping is done (given that Wikipedia is constantly changing).

You will need the `requests` and `BeautifulSoup` libraries for these questions, so let's import them in the following chunk. Also for your convenience, let's assign the URL of our page into a variable called "url"


In [5]:
import requests
from bs4 import BeautifulSoup
url = "https://en.wikipedia.org/w/index.php?title=Massachusetts&oldid=1263937561"



### Question 3 (1 point)

In the page, how many elements are there with an `<img>` tag? Write code below that finds the result and prints the answer. (For example, **if** the answer is 100, when I run the chunk below, it should print 100).

In [6]:
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

img_tags = soup.find_all("img")
print(len(img_tags))

84


### Question 4 (2 points)

Most elements with an `<img>` tag have `width` and `height` attributes that determine their display size on the page. Find the link to the source file of the image that has the biggest display area and print the link.



In [9]:
largest_area = 0
largest_img = None

for img in img_tags:
    width = int(img.get("width", 0))
    height = int(img.get("height", 0))
    area = width * height
    if area > largest_area:
        largest_area = area
        largest_img = img.get("src")

print(url + largest_img)

https://en.wikipedia.org/w/index.php?title=Massachusetts&oldid=1263937561//upload.wikimedia.org/wikipedia/commons/thumb/8/89/Massachusetts_population_map.png/350px-Massachusetts_population_map.png


## Question 5 (2 points)

Many of the links on the page are internal pages, which means they link to other pages in English Wikipedia. Internal links have an `href` attribute value that starts with the phrase `/wiki/`. For example, the `href` attribute value `/wiki/Greater_Boston` links to `https://en.wikipedia.org/wiki/Greater_Boston`. So, for our purposes, if an `href` value starts with `/wiki/` it is an internal link. How many internal links are there? Write code below that finds the result and prints the answer.




In [12]:
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    link_tags = soup.find_all('a')
    internal_link_count = sum(1 for link in link_tags if link.get('href', '').startswith('/wiki/'))
    print(internal_link_count)
else:
    print(f"Error fetching the page: {response.status_code}")

2659
